# Research Paper Analyzer — ML Evaluation Notebook

This notebook demonstrates the full evaluation workflow used inside the Research Paper Analyzer:
- **ROUGE scores** comparing LSA (extractive) vs BART (abstractive) summarization
- **Keyword method complementarity** between TF-IDF and Named Entity Recognition
- **Semantic coherence** of key findings using sentence embeddings

Run cells top-to-bottom. Set `PDF_PATH` in Cell 2 to point to any research paper PDF.

In [ ]:
# ── Configuration ─────────────────────────────────────────────
import os, sys

# Point this at any research paper PDF
PDF_PATH = "../sample.pdf"   # <-- change me

# Make sure the project root is on the path so we can import nlp.*
PROJECT_ROOT = os.path.abspath("..")
if PROJECT_ROOT not in sys.path:
    sys.path.insert(0, PROJECT_ROOT)

assert os.path.exists(PDF_PATH), f"PDF not found: {PDF_PATH}"
print(f"Using: {PDF_PATH}")

In [ ]:
# ── Imports ───────────────────────────────────────────────────
import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

from nlp.extractor import extract_text_from_pdf
from nlp.summarizer import extractive_summary, generate_summary
from nlp.keywords import extract_keywords
from nlp.evaluator import evaluate_summaries, evaluate_keywords, _extract_abstract
from nlp.embedder import embed_findings

import spacy
try:
    nlp = spacy.load("en_core_web_sm")
except OSError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "spacy", "download", "en_core_web_sm"])
    nlp = spacy.load("en_core_web_sm")

print("Imports OK")

In [ ]:
# ── Text extraction & abstract detection ──────────────────────
doc_info = extract_text_from_pdf(PDF_PATH)
text     = doc_info["text"]

print(f"Pages : {doc_info['page_count']}")
print(f"Words : {doc_info['word_count']:,}")
print(f"Read  : ~{doc_info['reading_time_min']} min\n")

abstract = _extract_abstract(text)
if abstract:
    print("Abstract detected (first 300 chars):")
    print(abstract[:300])
else:
    print("No abstract detected — will use mutual ROUGE fallback.")

In [ ]:
# ── Run both summarizers ───────────────────────────────────────
print("Running LSA extractive summary...")
lsa_summary = extractive_summary(text, sentence_count=12)
print(f"LSA summary ({len(lsa_summary.split())} words):\n{lsa_summary[:300]}...\n")

print("Running BART abstractive summary (may download model ~1.6 GB on first run)...")
bart_summary = generate_summary(text)
print(f"BART summary ({len(bart_summary.split())} words):\n{bart_summary[:300]}...")

In [ ]:
# ── Compute ROUGE scores ───────────────────────────────────────
from rouge_score import rouge_scorer

scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
ref    = abstract if abstract else bart_summary  # fallback

lsa_scores  = scorer.score(ref, lsa_summary)
bart_scores = scorer.score(ref, bart_summary)

print(f"{'Metric':<10} {'LSA P':>8} {'LSA R':>8} {'LSA F1':>8}   {'BART P':>8} {'BART R':>8} {'BART F1':>8}")
print("-" * 70)
for m in ["rouge1", "rouge2", "rougeL"]:
    l = lsa_scores[m]
    b = bart_scores[m]
    print(f"{m:<10} {l.precision:>8.3f} {l.recall:>8.3f} {l.fmeasure:>8.3f}   "
          f"{b.precision:>8.3f} {b.recall:>8.3f} {b.fmeasure:>8.3f}")

## Interpreting ROUGE

| Metric | Measures |
|--------|----------|
| **ROUGE-1** | Unigram overlap — how many individual words are shared |
| **ROUGE-2** | Bigram overlap — how many 2-word phrases are shared |
| **ROUGE-L** | Longest Common Subsequence — fluency-aware overlap |

**Precision** = fraction of generated words that appear in the reference.  
**Recall** = fraction of reference words that appear in the generated summary.  
**F1** = harmonic mean of precision and recall.

> ROUGE rewards n-gram overlap, not factual accuracy or readability. An extractive
> method that copies sentences verbatim will often score higher than an abstractive
> method that paraphrases — even if the abstractive summary is more useful.

In [ ]:
# ── ROUGE bar chart ────────────────────────────────────────────
metrics   = ["rouge1", "rouge2", "rougeL"]
sub_names = ["Precision", "Recall", "F1"]
lsa_vals  = [[getattr(lsa_scores[m],  k) for k in ["precision","recall","fmeasure"]] for m in metrics]
bart_vals = [[getattr(bart_scores[m], k) for k in ["precision","recall","fmeasure"]] for m in metrics]

fig, axes = plt.subplots(1, 3, figsize=(14, 4), sharey=True)
fig.suptitle("ROUGE Scores: LSA vs BART", fontsize=14, fontweight="bold")

teal   = "#2DD4BF"
indigo = "#818CF8"
x      = np.arange(len(sub_names))
w      = 0.35

for i, (ax, m) in enumerate(zip(axes, metrics)):
    ax.bar(x - w/2, lsa_vals[i],  w, label="LSA",  color=teal,   alpha=0.85)
    ax.bar(x + w/2, bart_vals[i], w, label="BART", color=indigo, alpha=0.85)
    ax.set_title(m.upper())
    ax.set_xticks(x)
    ax.set_xticklabels(sub_names)
    ax.set_ylim(0, 1)
    ax.set_ylabel("Score" if i == 0 else "")
    ax.legend()
    ax.spines[["top", "right"]].set_visible(False)

plt.tight_layout()
plt.savefig("rouge_comparison.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved rouge_comparison.png")

In [ ]:
# ── Keyword extraction ─────────────────────────────────────────
keywords = extract_keywords(text, nlp)

tfidf_kws = [kw for kw in keywords if kw["type"] == "tfidf"]
ner_kws   = [kw for kw in keywords if kw["type"] == "entity"]

print(f"TF-IDF keywords ({len(tfidf_kws)}):")
print(", ".join(k["keyword"] for k in tfidf_kws[:15]))
print(f"\nNER entities ({len(ner_kws)}):")
print(", ".join(k["keyword"] for k in ner_kws[:15]))

kw_eval = evaluate_keywords(keywords)
print(f"\nOverlap rate : {kw_eval['overlap_rate']:.1%}")
print(f"Complementarity: {kw_eval['complementarity']:.1%}")
print(f"\n{kw_eval['interpretation']}")

In [ ]:
# ── Venn-style keyword overlap chart ──────────────────────────
tfidf_set = {kw["keyword"].lower() for kw in tfidf_kws}
ner_set   = {kw["keyword"].lower() for kw in ner_kws}
overlap   = tfidf_set & ner_set
tfidf_only = tfidf_set - overlap
ner_only   = ner_set   - overlap

fig, ax = plt.subplots(figsize=(8, 4))
ax.set_xlim(0, 10)
ax.set_ylim(0, 5)
ax.axis("off")

c1 = plt.Circle((3.5, 2.5), 2.0, color=teal,   alpha=0.35, zorder=1)
c2 = plt.Circle((6.5, 2.5), 2.0, color=indigo, alpha=0.35, zorder=1)
ax.add_patch(c1)
ax.add_patch(c2)

ax.text(2.2, 2.5, f"TF-IDF only\n{len(tfidf_only)}", ha="center", va="center", fontsize=12, color="#0D9488", fontweight="bold")
ax.text(5.0, 2.5, f"Both\n{len(overlap)}",            ha="center", va="center", fontsize=11, color="#334155")
ax.text(7.8, 2.5, f"NER only\n{len(ner_only)}",        ha="center", va="center", fontsize=12, color="#6366F1", fontweight="bold")
ax.set_title("Keyword Method Overlap", fontsize=13, fontweight="bold", pad=10)

patch1 = mpatches.Patch(color=teal,   alpha=0.6, label="TF-IDF")
patch2 = mpatches.Patch(color=indigo, alpha=0.6, label="NER Entities")
ax.legend(handles=[patch1, patch2], loc="lower center", ncol=2, frameon=False)

plt.tight_layout()
plt.savefig("keyword_overlap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved keyword_overlap.png")

## Why Two Keyword Methods?

| Method | Mechanism | Strength |
|--------|-----------|----------|
| **TF-IDF** | Scores terms by frequency in this doc vs. corpus | Finds domain-specific jargon and topic words |
| **NER (spaCy)** | Pattern-matches named entities (people, orgs, places, etc.) | Captures proper nouns and structured concepts |

The two methods are **complementary** rather than redundant:
- TF-IDF finds *what the paper is about* in a statistical sense
- NER finds *who/what/where* the paper discusses

High complementarity (low overlap) means the combined keyword list gives richer, more complete coverage of the paper's content.

In [ ]:
# ── Sentence embeddings ────────────────────────────────────────
from nlp.summarizer import extract_key_findings
from sentence_transformers import SentenceTransformer

findings = extract_key_findings(text)
print(f"{len(findings)} key findings extracted.\n")
for i, f in enumerate(findings, 1):
    print(f"{i}. {f[:100]}")

print("\nLoading all-MiniLM-L6-v2 (downloads ~80 MB on first run)...")
model      = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model.encode(findings)
print(f"Embedding shape: {embeddings.shape}")

In [ ]:
# ── Cosine similarity heatmap ──────────────────────────────────
from sklearn.metrics.pairwise import cosine_similarity
import matplotlib.colors as mcolors

sim_matrix = cosine_similarity(embeddings)

# Build teal→indigo colormap
cmap = mcolors.LinearSegmentedColormap.from_list(
    "teal_indigo", ["#05070E", "#2DD4BF", "#818CF8"], N=256
)

n = len(findings)
fig, ax = plt.subplots(figsize=(max(6, n), max(5, n - 1)))
im = ax.imshow(sim_matrix, cmap=cmap, vmin=0, vmax=1)
plt.colorbar(im, ax=ax, label="Cosine similarity")

short_labels = [f"F{i+1}" for i in range(n)]
ax.set_xticks(range(n))
ax.set_yticks(range(n))
ax.set_xticklabels(short_labels)
ax.set_yticklabels(short_labels)

# Annotate cells
for i in range(n):
    for j in range(n):
        ax.text(j, i, f"{sim_matrix[i,j]:.2f}", ha="center", va="center",
                color="white" if sim_matrix[i,j] < 0.6 else "black", fontsize=8)

ax.set_title("Finding-to-Finding Cosine Similarity", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.savefig("coherence_heatmap.png", dpi=150, bbox_inches="tight")
plt.show()
print("Saved coherence_heatmap.png")

avg_coherence = (sim_matrix.sum() - np.trace(sim_matrix)) / (n * (n - 1))
print(f"\nAverage off-diagonal cosine similarity: {avg_coherence:.3f}")

## Summary & Takeaways

| Evaluation dimension | Method | Metric |
|----------------------|--------|--------|
| Summary quality | ROUGE-1/2/L vs abstract | F1 score (0–1) |
| Keyword coverage | TF-IDF ∩ NER overlap | Complementarity % |
| Finding coherence | all-MiniLM-L6-v2 embeddings | Avg cosine similarity |

### What this project demonstrates
- **End-to-end NLP pipeline**: PDF extraction → section detection → dual summarization → multi-method keyword extraction → citation parsing
- **Quantitative evaluation**: ROUGE scores against the paper's own abstract provide a principled, dataset-free evaluation signal
- **Method comparison**: Explicitly measuring LSA vs BART trade-offs shows system-level thinking, not just library usage
- **Semantic reasoning**: Sentence embeddings add a layer of meaning-aware evaluation beyond surface n-gram overlap

### Limitations & Future Work
- BART (`facebook/bart-large-cnn`) was fine-tuned on news articles, not scientific papers — a science-tuned model (e.g., `allenai/led-large-16384`) would likely score higher on academic PDFs
- ROUGE does not measure factual accuracy or hallucinations; BERTScore or factual consistency metrics (DAE, QAFactEval) would be better production metrics
- The NER pipeline uses `en_core_web_sm` — a domain-specific science/biomedical NER (e.g., SciSpaCy) would extract higher-quality entities
- Coherence clustering uses a fixed 0.70 threshold — this could be made adaptive (e.g., elbow method on the similarity distribution)